In [1]:
import dataclasses

import jax
jax.config.update("jax_disable_jit",  True)  # 全局生效 


from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [ ]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_fast_droid")

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)

# Delete the policy to free up memory.
# del policy

print("Actions shape:", result["actions"].shape)

In [2]:
config = _config.get_config("pi0_droid")
checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_droid")

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)

print("Actions shape:", result["actions"].shape)

token max len 48
prefix_attn_mask shape (1, 816, 816), prefix_attn_mask value [[[ True  True  True ... False False False]
  [ True  True  True ... False False False]
  [ True  True  True ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]] 
noisy_actions shape (1, 10, 32)
state token shape (1, 1, 1024)
timestamp [1.]
time_emb shape (1, 1024)
action time token shape (1, 10, 1024)
time 1.0, suffix_attn_mask shape (1, 11, 11), suffix_attn_mask [[[ True False False False False False False False False False False]
  [ True  True  True  True  True  True  True  True  True  True  True]
  [ True  True  True  True  True  True  True  True  True  True  True]
  [ True  True  True  True  True  True  True  True  True  True  True]
  [ True  True  True  True  True  True  True  True  True  True  True]
  [ True  True  True  True  True  True  True  True  True  True  True]
  [ True  True  True  Tru

2025-03-20 13:51:45.639084: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 128.00MiB (rounded to 134217728)requested by op 
2025-03-20 13:51:45.640361: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ****************************************************************************************************
E0320 13:51:45.640394  572724 pjrt_stream_executor_client.cc:3045] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 134217728 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 134217728 bytes.

In [5]:
example.keys()

dict_keys(['observation/exterior_image_1_left', 'observation/wrist_image_left', 'observation/joint_position', 'observation/gripper_position', 'prompt'])

In [7]:
policy

NameError: name 'policy' is not defined

In [6]:
example['observation/gripper_position']

array([0.34306253])

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

  0%|          | 0.00/11.2G [00:00<?, ?iB/s]

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)